In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv('steam_indie_merged_data.csv', engine='python')

In [3]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,61256,61257,61258,61259,61260,61261,61262,61263,61264,61265
appid,1623730,304930,105600,431960,291550,4000,252490,346110,413150,242760,...,988640,1424350,1785940,3591890,3451960,2656520,2145300,2571600,1589920,1513670
spy_name,Palworld,Unturned,Terraria,Wallpaper Engine,Brawlhalla,Garry's Mod,Rust,ARK: Survival Evolved,Stardew Valley,The Forest,...,Christmas Wonderland,Bunny Hill,COVEN,CombatBox,101 Cats in New Delhi,Mycelium Heaven,The Game Store,Meurtre Au Florian,Long Ago: A Puzzle Tale,Fono
owners,"50,000,000 .. 100,000,000","50,000,000 .. 100,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000","20,000,000 .. 50,000,000",...,"0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000"
positive,358266,506516,1373979,876898,314809,1122546,1071135,612177,872384,599805,...,2,136,156,1,0,33,6,7,15,17
negative,22443,48852,35494,17560,71647,37161,156649,117993,13811,27986,...,0,8,8,0,0,3,4,2,2,0
price_spy,2999,0,999,499,0,999,3999,989,1499,1999,...,349,399,1499,99,99,999,799,499,99,599
ccu,18028,10408,24580,91184,14169,18400,143870,22170,50662,2892,...,0,0,2,0,0,0,0,0,0,0
name_store,Palworld,Unturned,Terraria,Wallpaper Engine,Brawlhalla,Garry's Mod,Rust,ARK: Survival Evolved,Stardew Valley,The Forest,...,Christmas Wonderland,Bunny Hill,COVEN,CombatBox,101 Cats in New Delhi,Mycelium Heaven,The Game Store,Meurtre Au Florian,Long Ago: A Puzzle Tale,Fono
type,game,game,game,game,game,game,game,game,game,game,...,game,game,game,game,game,game,game,game,game,game
genres,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...","['Action', 'Adventure', 'Casual', 'Indie', 'Fr...","['Action', 'Adventure', 'Indie', 'RPG']","['Casual', 'Indie', 'Animation & Modeling', 'D...","['Action', 'Indie', 'Free To Play']","['Casual', 'Indie', 'Simulation']","['Action', 'Adventure', 'Indie', 'Massively Mu...","['Action', 'Adventure', 'Indie', 'Massively Mu...","['Indie', 'RPG', 'Simulation']","['Action', 'Adventure', 'Indie', 'Simulation']",...,"['Adventure', 'Indie']","['Action', 'Casual', 'Indie', 'Racing', 'Sports']","['Action', 'Adventure', 'Indie', 'Early Access']","['Action', 'Indie', 'Simulation', 'Strategy']","['Casual', 'Indie']","['Casual', 'Indie']","['Indie', 'Simulation']","['Adventure', 'Indie']","['Casual', 'Indie']","['Casual', 'Indie']"


In [4]:
import pandas as pd
import numpy as np
import ast

# 1) 파일 불러오기
df = pd.read_csv("steam_indie_merged_data.csv")

print(df.head())
print(df.shape)
print(df.columns.tolist())


# -----------------------------
# 2) 기본 복사
# -----------------------------
df_clean = df.copy()


# -----------------------------
# 3) 문자열 컬럼 정리
# -----------------------------
text_cols = ["spy_name", "name_store", "owners", "type", "genres", "release_date", "developers"]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("string").str.strip()


# -----------------------------
# 4) owners 분해
# 예: "50,000,000 .. 100,000,000"
# -----------------------------
df_clean["owners"] = df_clean["owners"].str.replace(",", "", regex=False)

df_clean["owners_low"] = df_clean["owners"].str.split(r"\.\.").str[0].str.strip()
df_clean["owners_high"] = df_clean["owners"].str.split(r"\.\.").str[1].str.strip()

df_clean["owners_low"] = pd.to_numeric(df_clean["owners_low"], errors="coerce")
df_clean["owners_high"] = pd.to_numeric(df_clean["owners_high"], errors="coerce")

df_clean["owners_mid"] = (df_clean["owners_low"] + df_clean["owners_high"]) / 2


# -----------------------------
# 5) 수치형 컬럼 변환
# -----------------------------
numeric_cols = ["appid", "positive", "negative", "price_spy", "ccu"]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")


# -----------------------------
# 6) 가격 달러 단위로 변환
# 예: 2999 -> 29.99
# -----------------------------
df_clean["price_usd"] = df_clean["price_spy"] / 100


# -----------------------------
# 7) 리뷰 파생변수
# -----------------------------
df_clean["review_total"] = df_clean["positive"].fillna(0) + df_clean["negative"].fillna(0)

df_clean["positive_ratio"] = np.where(
    df_clean["review_total"] > 0,
    df_clean["positive"] / df_clean["review_total"],
    np.nan
)

df_clean["negative_ratio"] = np.where(
    df_clean["review_total"] > 0,
    df_clean["negative"] / df_clean["review_total"],
    np.nan
)

df_clean["log_review_total"] = np.log1p(df_clean["review_total"])


# -----------------------------
# 8) genres 문자열 -> 실제 리스트
# -----------------------------
def parse_genres(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

df_clean["genres_list"] = df_clean["genres"].apply(parse_genres)


# -----------------------------
# 9) 장르 플래그 생성
# -----------------------------
def has_genre(genres, genre_name):
    if not isinstance(genres, list):
        return 0
    return int(genre_name in genres)

genre_targets = [
    "Action", "Adventure", "RPG", "Strategy",
    "Simulation", "Casual", "Free To Play", "Early Access", "Indie"
]

for g in genre_targets:
    col_name = "genre_" + g.lower().replace(" ", "_")
    df_clean[col_name] = df_clean["genres_list"].apply(lambda x: has_genre(x, g))


# -----------------------------
# 10) 날짜 처리
# -----------------------------
df_clean["release_date_parsed"] = pd.to_datetime(
    df_clean["release_date"],
    format="%d %b, %Y",
    errors="coerce"
)

today = pd.Timestamp.today().normalize()

df_clean["days_since_release"] = (today - df_clean["release_date_parsed"]).dt.days
df_clean["release_year"] = df_clean["release_date_parsed"].dt.year
df_clean["release_month"] = df_clean["release_date_parsed"].dt.month


# -----------------------------
# 11) 무료 게임 여부
# -----------------------------
df_clean["is_free"] = (df_clean["price_usd"] == 0).astype(int)


# -----------------------------
# 12) 가격 구간화
# -----------------------------
df_clean["price_band"] = pd.cut(
    df_clean["price_usd"],
    bins=[-1, 0, 5, 10, 20, 30, 60, 9999],
    labels=["Free", "0~5", "5~10", "10~20", "20~30", "30~60", "60+"]
)


# -----------------------------
# 13) 리뷰 규모 구간화
# -----------------------------
df_clean["review_band"] = pd.cut(
    df_clean["review_total"],
    bins=[-1, 10, 50, 100, 500, 1000, 10000, 999999999],
    labels=["0~10", "11~50", "51~100", "101~500", "501~1000", "1001~10000", "10000+"]
)


# -----------------------------
# 14) 이름 정리용 대표 이름 만들기
# store 이름 우선, 없으면 spy 이름 사용
# -----------------------------
df_clean["game_name"] = df_clean["name_store"].fillna(df_clean["spy_name"])


# -----------------------------
# 15) 최종 확인
# -----------------------------
print(df_clean.head())
print(df_clean.shape)
print(df_clean.columns.tolist())
print(df_clean.isna().sum().sort_values(ascending=False).head(20))


# -----------------------------
# 16) 저장
# -----------------------------
df_clean.to_csv("steam_indie_preprocessed.csv", index=False, encoding="utf-8-sig")
print("저장 완료: steam_indie_preprocessed.csv")

     appid          spy_name                     owners  positive  negative  \
0  1623730          Palworld  50,000,000 .. 100,000,000    358266     22443   
1   304930          Unturned  50,000,000 .. 100,000,000    506516     48852   
2   105600          Terraria   20,000,000 .. 50,000,000   1373979     35494   
3   431960  Wallpaper Engine   20,000,000 .. 50,000,000    876898     17560   
4   291550        Brawlhalla   20,000,000 .. 50,000,000    314809     71647   

   price_spy    ccu        name_store  type  \
0       2999  18028          Palworld  game   
1          0  10408          Unturned  game   
2        999  24580          Terraria  game   
3        499  91184  Wallpaper Engine  game   
4          0  14169        Brawlhalla  game   

                                              genres  release_date  \
0  ['Action', 'Adventure', 'Indie', 'RPG', 'Early...  18 Jan, 2024   
1  ['Action', 'Adventure', 'Casual', 'Indie', 'Fr...   7 Jul, 2017   
2            ['Action', 'Adventu

In [5]:
df_clean.T

,0,1,2,3,4,5,6,7,8,9,...,61256,61257,61258,61259,61260,61261,61262,61263,61264,61265
appid,1623730,304930,105600,431960,291550,4000,252490,346110,413150,242760,...,988640,1424350,1785940,3591890,3451960,2656520,2145300,2571600,1589920,1513670
spy_name,Palworld,Unturned,Terraria,Wallpaper Engine,Brawlhalla,Garry's Mod,Rust,ARK: Survival Evolved,Stardew Valley,The Forest,...,Christmas Wonderland,Bunny Hill,COVEN,CombatBox,101 Cats in New Delhi,Mycelium Heaven,The Game Store,Meurtre Au Florian,Long Ago: A Puzzle Tale,Fono
owners,50000000 .. 100000000,50000000 .. 100000000,20000000 .. 50000000,20000000 .. 50000000,20000000 .. 50000000,20000000 .. 50000000,20000000 .. 50000000,20000000 .. 50000000,20000000 .. 50000000,20000000 .. 50000000,...,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000,0 .. 20000
positive,358266,506516,1373979,876898,314809,1122546,1071135,612177,872384,599805,...,2,136,156,1,0,33,6,7,15,17
negative,22443,48852,35494,17560,71647,37161,156649,117993,13811,27986,...,0,8,8,0,0,3,4,2,2,0
price_spy,2999,0,999,499,0,999,3999,989,1499,1999,...,349,399,1499,99,99,999,799,499,99,599
ccu,18028,10408,24580,91184,14169,18400,143870,22170,50662,2892,...,0,0,2,0,0,0,0,0,0,0
name_store,Palworld,Unturned,Terraria,Wallpaper Engine,Brawlhalla,Garry's Mod,Rust,ARK: Survival Evolved,Stardew Valley,The Forest,...,Christmas Wonderland,Bunny Hill,COVEN,CombatBox,101 Cats in New Delhi,Mycelium Heaven,The Game Store,Meurtre Au Florian,Long Ago: A Puzzle Tale,Fono
type,game,game,game,game,game,game,game,game,game,game,...,game,game,game,game,game,game,game,game,game,game
genres,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...","['Action', 'Adventure', 'Casual', 'Indie', 'Fr...","['Action', 'Adventure', 'Indie', 'RPG']","['Casual', 'Indie', 'Animation & Modeling', 'D...","['Action', 'Indie', 'Free To Play']","['Casual', 'Indie', 'Simulation']","['Action', 'Adventure', 'Indie', 'Massively Mu...","['Action', 'Adventure', 'Indie', 'Massively Mu...","['Indie', 'RPG', 'Simulation']","['Action', 'Adventure', 'Indie', 'Simulation']",...,"['Adventure', 'Indie']","['Action', 'Casual', 'Indie', 'Racing', 'Sports']","['Action', 'Adventure', 'Indie', 'Early Access']","['Action', 'Indie', 'Simulation', 'Strategy']","['Casual', 'Indie']","['Casual', 'Indie']","['Indie', 'Simulation']","['Adventure', 'Indie']","['Casual', 'Indie']","['Casual', 'Indie']"


In [6]:
print(df_clean[[
    "price_usd",
    "review_total",
    "positive_ratio",
    "owners_mid",
    "days_since_release"
]].describe())

          price_usd  review_total  positive_ratio    owners_mid  \
count  61266.000000  6.126600e+04    60897.000000  6.126600e+04   
mean       6.560010  9.424468e+02        0.760991  7.515278e+04   
std       11.142416  1.509816e+04        0.235797  7.309981e+05   
min        0.000000  0.000000e+00        0.000000  1.000000e+04   
25%        1.090000  6.000000e+00        0.652605  1.000000e+04   
50%        4.990000  2.300000e+01        0.820513  1.000000e+04   
75%        9.990000  1.040000e+02        0.946150  3.500000e+04   
max      999.980000  1.409473e+06        1.000000  7.500000e+07   

       days_since_release  
count        61109.000000  
mean          1924.746093  
std           1119.609589  
min          -1296.000000  
25%            944.000000  
50%           1829.000000  
75%           2769.000000  
max          10522.000000  


In [7]:
#가격별 리뷰수
print(
    df_clean.groupby("price_band", observed=False)["review_total"]
    .mean()
    .sort_values(ascending=False)
)

price_band
30~60    14142.108014
20~30     5415.794047
10~20     2176.538604
Free      1078.451786
5~10       898.460475
0~5        274.736028
60+        119.140496
Name: review_total, dtype: float64


In [8]:
#장르별 리뷰량
genre_cols = [
    "genre_action", "genre_adventure", "genre_rpg",
    "genre_strategy", "genre_simulation", "genre_casual"
]

for col in genre_cols:
    print(f"\n[{col}]")
    print(df_clean.groupby(col)["review_total"].mean())


[genre_action]
genre_action
0     748.241356
1    1170.927457
Name: review_total, dtype: float64

[genre_adventure]
genre_adventure
0     853.500578
1    1057.964768
Name: review_total, dtype: float64

[genre_rpg]
genre_rpg
0     808.606587
1    1514.213235
Name: review_total, dtype: float64

[genre_strategy]
genre_strategy
0    947.113287
1    924.250700
Name: review_total, dtype: float64

[genre_simulation]
genre_simulation
0     792.466579
1    1520.918186
Name: review_total, dtype: float64

[genre_casual]
genre_casual
0    1253.809690
1     550.600398
Name: review_total, dtype: float64


In [9]:
print(df_clean.shape)
print('-'*50)
print(df_clean.columns.tolist())
print('-'*50)
print(df_clean.head())
print('-'*50)
print(df_clean.info())

(61266, 38)
--------------------------------------------------
['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers', 'owners_low', 'owners_high', 'owners_mid', 'price_usd', 'review_total', 'positive_ratio', 'negative_ratio', 'log_review_total', 'genres_list', 'genre_action', 'genre_adventure', 'genre_rpg', 'genre_strategy', 'genre_simulation', 'genre_casual', 'genre_free_to_play', 'genre_early_access', 'genre_indie', 'release_date_parsed', 'days_since_release', 'release_year', 'release_month', 'is_free', 'price_band', 'review_band', 'game_name']
--------------------------------------------------
     appid          spy_name                 owners  positive  negative  \
0  1623730          Palworld  50000000 .. 100000000    358266     22443   
1   304930          Unturned  50000000 .. 100000000    506516     48852   
2   105600          Terraria   20000000 .. 50000000   1373979     35494   
3   431960  W

In [10]:
na_summary = df_clean.isna().sum().sort_values(ascending=False)
print(na_summary.head(20))

positive_ratio         369
negative_ratio         369
release_month          157
release_year           157
days_since_release     157
release_date_parsed    157
developers              96
release_date            33
spy_name                 8
name_store               3
game_name                2
appid                    0
positive                 0
owners                   0
price_spy                0
negative                 0
price_usd                0
owners_low               0
owners_high              0
type                     0
dtype: int64


In [11]:
df_clean['positive'].isna().sum()

np.int64(0)

In [12]:
df_clean['positive_ratio'].isna().sum()

np.int64(369)

In [13]:
df_clean['negative'].isna().sum()

np.int64(0)

In [14]:
df_clean['negative_ratio'].isna().sum()

np.int64(369)

In [15]:
df_clean.sort_values(by='review_total', ascending=False).head()

,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,...,genre_early_access,genre_indie,release_date_parsed,days_since_release,release_year,release_month,is_free,price_band,review_band,game_name
2,105600,Terraria,20000000 .. 50000000,1373979,35494,999,24580,Terraria,game,"['Action', 'Adventure', 'Indie', 'RPG']",...,0,1,2011-05-16,5454.0,2011.0,5.0,0,5~10,10000+,Terraria
6,252490,Rust,20000000 .. 50000000,1071135,156649,3999,143870,Rust,game,"['Action', 'Adventure', 'Indie', 'Massively Mu...",...,0,1,2018-02-08,2994.0,2018.0,2.0,0,30~60,10000+,Rust
5,4000,Garry's Mod,20000000 .. 50000000,1122546,37161,999,18400,Garry's Mod,game,"['Casual', 'Indie', 'Simulation']",...,0,1,2006-11-29,7083.0,2006.0,11.0,0,5~10,10000+,Garry's Mod
3,431960,Wallpaper Engine,20000000 .. 50000000,876898,17560,499,91184,Wallpaper Engine,game,"['Casual', 'Indie', 'Animation & Modeling', 'D...",...,0,1,2018-11-16,2713.0,2018.0,11.0,0,0~5,10000+,Wallpaper Engine
8,413150,Stardew Valley,20000000 .. 50000000,872384,13811,1499,50662,Stardew Valley,game,"['Indie', 'RPG', 'Simulation']",...,0,1,NaT,NaN,NaN,NaN,0,10~20,10000+,Stardew Valley


In [16]:
df_clean[df_clean['review_total'] > 30].sort_values(by='review_total', ascending=True).head()

,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,...,genre_early_access,genre_indie,release_date_parsed,days_since_release,release_year,release_month,is_free,price_band,review_band,game_name
16821,1504310,Azure Sky,20000 .. 50000,19,12,399,0,Azure Sky,game,"['Action', 'Adventure', 'Casual', 'Indie', 'RPG']",...,0,1,2021-01-25,1912.0,2021.0,1.0,0,0~5,11~50,Azure Sky
16810,650880,Beyond the Invisible: Evening,20000 .. 50000,18,13,199,0,Beyond the Invisible: Evening,game,"['Adventure', 'Casual', 'Indie']",...,0,1,2017-06-22,3225.0,2017.0,6.0,0,0~5,11~50,Beyond the Invisible: Evening
20017,1412070,Siege the Day,0 .. 20000,16,15,1299,0,Siege the Day,game,"['Action', 'Indie', 'Strategy']",...,0,1,2023-12-21,852.0,2023.0,12.0,0,10~20,11~50,Siege the Day
31735,2895110,Dungeon Deck,0 .. 20000,31,0,999,2,Dungeon Deck,game,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",...,0,1,2024-11-28,509.0,2024.0,11.0,0,5~10,11~50,Dungeon Deck
31830,765590,Co-op SNEK Online,0 .. 20000,27,4,299,0,Co-op SNEK Online,game,"['Action', 'Casual', 'Indie']",...,0,1,2018-04-13,2930.0,2018.0,4.0,0,0~5,11~50,Co-op SNEK Online


In [17]:
df_clean["genres_list"].dtype

dtype('O')

In [18]:
all_genres = df_clean["genres_list"].explode().dropna()

print(all_genres.unique())
print("장르 개수:", len(all_genres.unique()))

<ArrowStringArray>
[               'Action',             'Adventure',                 'Indie',
                   'RPG',          'Early Access',                'Casual',
          'Free To Play',  'Animation & Modeling', 'Design & Illustration',
         'Photo Editing',             'Utilities',            'Simulation',
 'Massively Multiplayer',                'Sports',              'Strategy',
                'Racing',     'Software Training',               'Violent',
      'Video Production',             'Education',      'Game Development',
                'Nudity',        'Web Publishing',                 'Movie',
                  'Gore',      'Audio Production',        'Sexual Content',
            'Accounting',                 'Short']
Length: 29, dtype: str
장르 개수: 29


In [19]:
genre_counts = all_genres.value_counts()

genre_counts

genres_list
Indie                    61218
Action                   28149
Casual                   27127
Adventure                26652
Simulation               12614
Strategy                 12505
RPG                      11621
Early Access              6340
Free To Play              3446
Sports                    2514
Racing                    2220
Massively Multiplayer     1233
Violent                    332
Gore                       214
Utilities                   80
Nudity                      75
Sexual Content              71
Education                   46
Design & Illustration       43
Animation & Modeling        41
Game Development            38
Audio Production            30
Video Production            26
Software Training           23
Photo Editing               18
Web Publishing              11
Accounting                   6
Movie                        1
Short                        1
Name: count, dtype: int64

In [20]:
df_2024_after = df_clean[df_clean["release_date_parsed"] >= "2024-01-01"].copy()
print("-"*50)
print(df_2024_after.shape)
print("-"*50)
print(df_2024_after.head())
print("-"*50)
print(df_2024_after.tail())

--------------------------------------------------
(13221, 38)
--------------------------------------------------
      appid       spy_name                 owners  positive  negative  \
0   1623730       Palworld  50000000 .. 100000000    358266     22443   
10   899770     Last Epoch   20000000 .. 50000000     88027     22596   
20  3164500     Schedule I   10000000 .. 20000000    200803      3238   
23   251570  7 Days to Die   10000000 .. 20000000    327889     42157   
24  1116170      CyberCorp   10000000 .. 20000000       266        56   

    price_spy    ccu     name_store  type  \
0        2999  18028       Palworld  game   
10       3499   5831     Last Epoch  game   
20       1999  22757     Schedule I  game   
23       4499  17045  7 Days to Die  game   
24       1499      3      CyberCorp  game   

                                               genres  ... genre_early_access  \
0   ['Action', 'Adventure', 'Indie', 'RPG', 'Early...  ...                  1   
10            

In [21]:
genre_counts = all_genres.value_counts()

genre_counts

genres_list
Indie                    61218
Action                   28149
Casual                   27127
Adventure                26652
Simulation               12614
Strategy                 12505
RPG                      11621
Early Access              6340
Free To Play              3446
Sports                    2514
Racing                    2220
Massively Multiplayer     1233
Violent                    332
Gore                       214
Utilities                   80
Nudity                      75
Sexual Content              71
Education                   46
Design & Illustration       43
Animation & Modeling        41
Game Development            38
Audio Production            30
Video Production            26
Software Training           23
Photo Editing               18
Web Publishing              11
Accounting                   6
Movie                        1
Short                        1
Name: count, dtype: int64